# Analisi linea umana (MCF10 EV, MCF10 RAB5) — unità = cellula

Nel monostrato ogni curva è acquisita su una **cellula diversa** (6–7 curve per area, una per cellula):
**ogni curva è una cellula**. La sottocartella dell'area resta solo come etichetta e non viene usata per fare medie.
- **Forza-indentazione**: ogni curva è fittata con Ting ed è un'unità a sé.
- **Stress-relaxation**: ogni curva è normalizzata e fittata da sola (SLS, PLR, bi-exp, poroelastico).

Alla fine: tabella per cellula, curva SR media di ciascuna linea, test di Mann–Whitney tra le due linee (una osservazione per cellula).

**Da modificare:** solo la cella 2 (cartelle delle due linee).

**Nota:** gli output già salvati precedono la rampa di chiusura al contatto e i nuovi criteri di esclusione. Rieseguire la cella di analisi completa per aggiornare i risultati in `aggregated/per_cellula_umane_chiusura_contatto`; l'esclusione del 10/06/26 per EV e RAB5 resta attiva.

In [ ]:
# 1. Import
import sys
sys.path.insert(0, r'/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo')
sys.path.insert(0, r'/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/general/src')

import importlib
import warnings
import matplotlib.pyplot as plt
import analisi_per_cellula as apc
importlib.reload(apc)

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

In [ ]:
# 2. Configurazione
SISTEMA = "umane"

# Per ogni linea: le cartelle delle giornate. Dentro ciascuna: <area>/FD e <area>/SR
LINEE = {
    "EV": [
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_EV_090626",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_EV_100626",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_EV_130526",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_EV_140526",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_EV_250626",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_EV_260626",
        # aggiungi qui le altre giornate EV
    ],
    "RAB5": [
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_090626",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_100626",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_150626",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_160626",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_250626",
        r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/MCF10_RAB5_260626",
        # aggiungi qui le altre giornate RAB5
    ],
}

OUTPUT_DIR = r"/Users/furbetta/Desktop/PyFMlab_DyNaMo/PyFMGUI_DyNaMo/aggregated/per_cellula_umane_chiusura_contatto"
ESEGUI_FD = True
ESEGUI_SR = True
ESCLUDI_GIORNATE = ["100626"]
CRITERI = dict(apc.CRITERI_UMANE)
CRITERI["sr_F_relax_fisico"] = False
OPZIONI_TING = {"drag_mode": "minimo_cartella", "param_overrides": {"apply_ramp_correction": True}}
OPZIONI_SR = {"drag_correction": True}
ALTEZZE_UM = {"EV": 7.0, "RAB5": 7.0}

In [ ]:
# 3. Controllo: quali cellule vengono trovate (nessun fit)
cells = apc.find_cells(LINEE, SISTEMA)
apc.print_cells_overview(cells)
display(cells[["linea", "giornata", "cartella", "tipo", "id_cellula", "n_curve"]].head(20))

In [ ]:
# 4. Analisi completa: FD (Ting) e SR, tabelle per cellula, medie per linea, test
risultati = apc.run_full_analysis(
    LINEE,
    SISTEMA,
    OUTPUT_DIR,
    run_fd=ESEGUI_FD,
    run_sr=ESEGUI_SR,
    criteri=CRITERI,
    escludi_giornate=ESCLUDI_GIORNATE,
    ting_options=OPZIONI_TING,
    sr_options=OPZIONI_SR,
    altezze_um=ALTEZZE_UM,
)

In [ ]:
# 5. Indentazione durante il mantenimento (curve di stress-relaxation)
# Calcola delta(t) durante la pausa per ogni curva SR e la sua variazione relativa massima.
import os
import pandas as pd
import sr_utils as sr
importlib.reload(sr)

cells_sr = apc.find_cells(LINEE, SISTEMA)
cells_sr = cells_sr[cells_sr["tipo"] == "SR"]
if globals().get("ESCLUDI_GIORNATE"):
    cells_sr = cells_sr[~cells_sr["giornata"].astype(str).str.contains("|".join(ESCLUDI_GIORNATE))]

tabelle = []
ris_tesi = {}
for linea, gruppo in cells_sr.groupby("linea", sort=False):
    files = [f for lista in gruppo["curve"] for f in lista]
    nome = apc.NOMI_BREVI_TESI.get(linea, linea)
    print(f"--- {nome}")
    tab, ris = sr.variazione_indentazione_pausa(files)
    tab.insert(0, "linea", nome)
    tabelle.append(tab)
    ris_tesi[nome] = ris
    sr.grafico_indentazione_pausa(ris, titolo=nome,
                                  path_pdf=os.path.join(OUTPUT_DIR, f"delta_pausa_{SISTEMA}_{nome}.pdf"))
    plt.show()

sr.figura_tesi_indentazione_pausa(ris_tesi, os.path.join(OUTPUT_DIR, f"fig_tesi_delta_pausa_{SISTEMA}.pdf"))
plt.show()

tab_delta = pd.concat(tabelle, ignore_index=True)
tab_delta.to_csv(os.path.join(OUTPUT_DIR, f"delta_pausa_{SISTEMA}.csv"), index=False)
display(tab_delta[tab_delta["valida"]].groupby("linea")[["delta_0_um", "variazione_max_nm", "variazione_relativa_max_pct"]]
        .agg(["mean", "std", "median", "max"]).round(2))